# The pipeline, and moving it home

The prediction jobs are moving from Cloud Functions (run by Cloud Scheduler) to this homelab. Each job has a **mode**:

| mode | what it does |
|---|---|
| **off** | the timer fires and does nothing. Every job starts here. |
| **dry** | runs on its schedule and does all its work, but writes nothing. (A few jobs refuse a dry run; for those it is a preflight only.) |
| **live** | runs and writes to production BigQuery. |

This notebook only **reads**. Changing a mode is done in the control app (`Pipeline`), which refuses to go live while the job's Cloud Scheduler twin is still enabled.

In [ ]:
from mllab import nb
pl = nb.pipeline()
print("as of", pl.attrs.get("taken_at"))
pl

In [ ]:
pl.groupby(["bundle", "mode"]).size().unstack(fill_value=0)

Anything that is on and whose last run failed:

In [ ]:
pl[(pl["mode"].isin(["dry", "live"])) & (pl["last_ok"] == False)]   # noqa: E712

## Moving one job to live

Do these in order, one job at a time. The control app enforces the check in step 2.

1. **Dry run for a few cycles.** Set the job to *Dry run* in the app, let it run on schedule, and read the log (click the job). It should do the work and write nothing.
2. **Pause its Cloud Scheduler twin**, from your laptop (audited there): `mllab control scheduler pause <job> --confirm <job>`. The app shows the exact command.
3. **Set it to Live** in the app. It re-checks Cloud Scheduler, then asks you to type the job's name.

Per-game MLB (`pregame`) is the exception: it is created from **two** places (the Scheduler job `mlb-2026-pregame-schedule` and the App Engine cron entry
`/api/lineup/schedule-today`), and its Cloud Tasks queues must be drained. The app lists every step and asks you to acknowledge each.

The Cloud Functions stay deployed, idle, as a manual fallback: there is no automatic failover, because two writers on one table is worse than one missed run.

## What the model control plane does

Separate from the pipeline: it decides what the **site** shows and which model the pipeline runs, without a code change (hide a model, pin an exact artifact,
freeze a sport with a banner). Its settings live in BigQuery's `control` dataset and are changed with `mllab control …` from your laptop. See `docs/MODEL_CONTROL.md`.

In [ ]:
nb.app()